## What this notebook does

In this notebook, we:
- Load the datasets created/used in the EDA notebook
- Merge movies and ratings
- Create behaviour-based features (previous genre)
- Convert text data into numbers
- Add simple but useful features
- Save the final dataset for model training

This notebook focuses on **clarity and understanding**, not complicated logic.

## Step 1: Import Libraries
We first import the basic libraries needed for data preprocessing.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import pickle

## Step 2: Load the Datasets
We load the two main files:
- movies.csv → contains movieId, title and genres
- ratings.csv → contains user ratings and timestamps

In [ ]:
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

print("Movies shape:", movies.shape)
print("Ratings shape:", ratings.shape)

movies.head()

Movies shape: (9742, 3)
Ratings shape: (100836, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## Step 3: Merge Movies and Ratings
We combine both datasets using movieId so that each rating also contains the movie genre.

In [ ]:
data = pd.merge(ratings, movies, on="movieId")
data.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


## Step 4: Extract Primary Genre
Each movie may contain multiple genres. To keep the model simple, we take the first genre as the primary genre.

In [ ]:
data['primary_genre'] = data['genres'].apply(lambda x: x.split('|')[0])
data.head()

,userId,movieId,rating,timestamp,title,genres,primary_genre
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance,Comedy
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller,Action
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,Mystery
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,Crime


## Step 5: Sort Data Based on User Behaviour
We sort the data by userId and timestamp so that we can understand the sequence of movies watched by each user.

In [ ]:
data = data.sort_values(['userId', 'timestamp']).reset_index(drop=True)
data.head()

,userId,movieId,rating,timestamp,title,genres,primary_genre
0,1,804,4.0,964980499,She's the One (1996),Comedy|Romance,Comedy
1,1,1210,5.0,964980499,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,Action
2,1,2018,5.0,964980523,Bambi (1942),Animation|Children|Drama,Animation
3,1,2628,4.0,964980523,Star Wars: Episode I - The Phantom Menace (1999),Action|Adventure|Sci-Fi,Action
4,1,2826,4.0,964980523,"13th Warrior, The (1999)",Action|Adventure|Fantasy,Action


## Step 6: Create Previous Genre Feature
We create a new feature called prev_genre which tells us what genre the user watched before the current movie.

In [ ]:
data['prev_genre'] = data.groupby('userId')['primary_genre'].shift(1)
data['prev_genre'].fillna("None", inplace=True)
data.head()

/tmp/ipykernel_6732/3357713303.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['prev_genre'].fillna("None", inplace=True)


,userId,movieId,rating,timestamp,title,genres,primary_genre,prev_genre
0,1,804,4.0,964980499,She's the One (1996),Comedy|Romance,Comedy,None
1,1,1210,5.0,964980499,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,Action,Comedy
2,1,2018,5.0,964980523,Bambi (1942),Animation|Children|Drama,Animation,Action
3,1,2628,4.0,964980523,Star Wars: Episode I - The Phantom Menace (1999),Action|Adventure|Sci-Fi,Action,Animation
4,1,2826,4.0,964980523,"13th Warrior, The (1999)",Action|Adventure|Fantasy,Action,Action


## Step 7: Convert Genres into Numbers
Machine learning models cannot understand text, so we convert genres into numbers using Label Encoding.

In [ ]:
le1 = LabelEncoder()
le2 = LabelEncoder()

data['genre_encoded'] = le1.fit_transform(data['primary_genre'])
data['prev_genre_encoded'] = le2.fit_transform(data['prev_genre'])

data.head()

,userId,movieId,rating,timestamp,title,genres,primary_genre,prev_genre,genre_encoded,prev_genre_encoded
0,1,804,4.0,964980499,She's the One (1996),Comedy|Romance,Comedy,None,5,14
1,1,1210,5.0,964980499,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,Action,Comedy,1,5
2,1,2018,5.0,964980523,Bambi (1942),Animation|Children|Drama,Animation,Action,3,1
3,1,2628,4.0,964980523,Star Wars: Episode I - The Phantom Menace (1999),Action|Adventure|Sci-Fi,Action,Animation,1,3
4,1,2826,4.0,964980523,"13th Warrior, The (1999)",Action|Adventure|Fantasy,Action,Action,1,1


## Step 8: Create Simple Behaviour Features
We now create two useful features:
- Average rating given by each user
- Average rating received by each movie

In [ ]:
user_avg = data.groupby('userId')['rating'].mean()
movie_avg = data.groupby('movieId')['rating'].mean()

data['user_avg_rating'] = data['userId'].map(user_avg)
data['movie_avg_rating'] = data['movieId'].map(movie_avg)

data.head()

,userId,movieId,rating,timestamp,title,genres,primary_genre,prev_genre,genre_encoded,prev_genre_encoded,user_avg_rating,movie_avg_rating
0,1,804,4.0,964980499,She's the One (1996),Comedy|Romance,Comedy,None,5,14,4.366379,3.250000
1,1,1210,5.0,964980499,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,Action,Comedy,1,5,4.366379,4.137755
2,1,2018,5.0,964980523,Bambi (1942),Animation|Children|Drama,Animation,Action,3,1,4.366379,3.368421
3,1,2628,4.0,964980523,Star Wars: Episode I - The Phantom Menace (1999),Action|Adventure|Sci-Fi,Action,Animation,1,3,4.366379,3.107143
4,1,2826,4.0,964980523,"13th Warrior, The (1999)",Action|Adventure|Fantasy,Action,Action,1,1,4.366379,2.903846


## Step 9: Save the Final Dataset
We save the cleaned dataset so that we can directly use it in the model training notebook.

In [ ]:
data.to_csv("processed_data.csv", index=False)

with open("encoders.pkl", "wb") as f:
    pickle.dump((le1, le2), f)

print("Dataset saved successfully!")

Dataset saved successfully!
